# Example 1 — Force Sampling RDF Calculation

Welcome to revelsMD a one stop shot for reduced variance sampling using various techiniques based around the usage of force densities.

In [ ]:
from revelsMD.rdf import RDF, compute_rdf
from revelsMD.trajectories import LammpsTrajectory
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

First we generate a trajectory object, the trajectory is a python object which defines the necessary information about a trajectory required for parsing. RevelsMD can operate on a number of different trajecotry types flexing across core MD codes (lammps and gromacs), aimd dft codes (vasp), and a slightly more involved numpy driven version. In order to be able to parse forces and positions from each of these codes different parsers are used for each software and the Trajectory class allows this to be flexibly under the hood.

In [ ]:
traj=LammpsTrajectory('../../../examples/example_1_LJ/dump.nh.lammps','../../../examples/example_1_LJ/data.fin.nh.data',units='lj',atom_style="id resid type q x y z ix iy iz")

Then we feed it to a run function if your using lammps data as in this case you will need to assert if you are using lj, real, or metal units.

## Calculating an rdf from infinity 

We can then run a radial distribution function using RevelsMD. As described here (https://aip.scitation.org/doi/abs/10.1063/5.0053737) we have the ability to caluclate three different ways, by taking our Heaviside function from: infinity to zero, zero to inifity, or performing a linear combination of the two. Here we start by using the oldest formulation (from inifity) developed by Borgis.

In [ ]:
rdf_result=compute_rdf(traj,'1','1',period=1,delr=0.005,integration='backward')

In [ ]:
help(compute_rdf)

In [ ]:
plt.figure(figsize=(4,4))
plt.plot(rdf_result.r,rdf_result.g)
plt.ylabel('g(r)',size=16)
plt.xlabel(r'r/ $\operatorname{\AA}$',size=16)

One of the adventages of this family of methods is that a continuous if often noisy instantaneous (single frame) radial distribution function can be calculated. We can obtain this by setting the period to the number of frames in the the trajectory state.

In [ ]:
rdf_instantaneous=compute_rdf(traj,'1','1',period=int(traj.frames),delr=0.005,integration='backward')

In [ ]:
plt.figure(figsize=(4,4))
plt.plot(rdf_result.r,rdf_result.g)
plt.plot(rdf_instantaneous.r,rdf_instantaneous.g,color='red',linestyle='dashed')
plt.ylabel('g(r)',size=16)
plt.xlabel(r'r/ $\operatorname{\AA}$',size=16)

Here this is exceptional agreement, for systems with less particles or where the system is solvate, it will be much worse, but it is important to remember that even if the graph is visulaly unappealing an instantaneous calculation has utility for the calculation of instantaneous thermodynmaic quantatities.

## Calculating an rdf from zero 

In comparison to conventional methodolgies force integration will not always recover directly both the zero and infinite limits, there is therefore a choice of whether to obtain a small (often inperceptable) error in the zero or large r limit. We chose to obtain the zero limit correctly with g(r)=0 as the default, this is as an infinite limit of 1 is not strictly always required, nor as aphysical as allowing overlap of particles up to r=0. 

From the perspective of the code this means that if from_zero is not set to false the calculation will be performed with Heaviside function taken from zero.

In [ ]:
rdf_default=compute_rdf(traj,'1','1',period=1,delr=0.005)

In [ ]:
plt.figure(figsize=(4,4))
plt.plot(rdf_default.r,rdf_default.g,color='purple')
plt.ylabel('g(r)',size=16)
plt.xlabel(r'r/ $\operatorname{\AA}$',size=16)

## Using a linear combination of these terms

The linear combination method described in https://aip.scitation.org/doi/abs/10.1063/5.0053737 has a way of removing the issue of incorrect limits, by taking a linear combination. This can be obtained by using compute_rdf with integration='lambda', which has the same inputs as the conventional rdf calculation.

In [ ]:
rdf_lambda=compute_rdf(traj,'1','1',period=1,delr=0.005,integration='lambda')

The output of this function is an RDF object with properties .r for bin centres, .g for the refined g(r) values, and .lam for the linear combination described in the paper. 

The rdf is indistinguishable for the number of particles in this example. But the limits are now correct.

In [ ]:
plt.figure(figsize=(4,4))
plt.plot(rdf_lambda.r,rdf_lambda.g,color='grey')
plt.ylabel('g(r)',size=16)
plt.xlabel(r'r/ $\operatorname{\AA}$',size=16)

The linear combination giving that rdf is shown below, the form will vary between systems, it'll always be 1 at r=0, and zero at r=infinity. As a general rule, the value will normaly be close to 0 at intermediate values to, as the first peak will have the greatest affect on varience.

In [ ]:
plt.figure(figsize=(4,4))
plt.plot(rdf_lambda.r,rdf_lambda.lam,color='darkGreen')
plt.ylabel(r'$\lambda$(r)',size=16)
plt.xlabel(r'r/ $\operatorname{\AA}$',size=16)